# Deploy Claims Gateway (GW1) & Interceptors

Deploy the AgentCore **Claims Gateway (GW1)** with its REQUEST and RESPONSE interceptor Lambdas.

**What the interceptor pattern is.** The gateway runs a Lambda on every call that inspects the caller's JWT and enforces their identity in two ways:

- **Tool access (authorization).** The REQUEST interceptor reads the group claim (`cognito:groups` on Cognito, `groups` on Okta), looks the group up in a DynamoDB role-map, and **403s any tool the group isn't allowed to call**; the RESPONSE interceptor filters the `tools/list` result so a caller only *sees* their permitted tools.
- **Data access (column + row security).** For allowed calls, the interceptor maps the group → a tenant **IAM role** and assumes it, so the downstream Claims MCP queries Athena / S3 Tables under that role: **Lake Formation** applies that role's column filtering and table grants, and the claims tools bind the propagated caller identity into a `WHERE user_id = ?` predicate for row scope.

**Why choose it.** When identity must drive *custom* policy — mapping an IdP group to a tenant IAM role, applying Lake Formation governance, or gating which tools a role may invoke — the interceptor Lambda is where that logic lives.

**Contrast with the OBO path (`05b`).** On the **Okta** path, GW2 propagates identity *natively*: AgentCore Identity exchanges the user's token (RFC 8693) for a re-scoped, per-user token and the notes store filters per-user (`owner_user_sub`) — with **no interceptor Lambda**. Same goal (the user's identity reaches the data), two mechanisms: **custom interceptor code here** vs **token exchange there**. The different data stores (Athena vs OpenSearch) make "which pattern am I using" tangible. On the **Cognito** path GW2 instead uses a thin notes REQUEST interceptor, since OBO token exchange is Okta-only in this sample.

GW1 is the claims path and is deployed on **both** IdP paths (Cognito and Okta) — the topology is identical. The authorizer (Cognito user-pool `allowedClients` vs Okta discovery `allowedAudience`) and the interceptor attachment are flag-branched **inside `create_gateway.py`** per `IDP_PROVIDER`, so the cells below stay IdP-agnostic.

## Prerequisites

- ✅ Run `04-deploy-mcp-server.ipynb` first
- ✅ MCP Server Runtime ARN saved to SSM

## What This Notebook Does

1. Deploys the claims **REQUEST** interceptor Lambda — validates the JWT and extracts caller identity (both IdP paths)
2. Deploys the claims **RESPONSE** interceptor Lambda — row/field filtering (per-persona `tools/list` + result scoping)
3. Creates the AgentCore Claims Gateway (GW1) and attaches both interceptors
4. Saves the Gateway ARN to SSM

> Note: the claims **REQUEST** interceptor uses a DynamoDB tenant-role map (`lakehouse_tenant_role_map`) to map the caller's group claim to an IAM role (tenant→role STS AssumeRole) and to gate tools per group (disallowed tool → HTTP 403). That table is created by `interceptor-request/setup_dynamodb_tenant_role_maps.py` (run by `interceptor-request/deploy.sh`), not by this notebook — this notebook wires the gateway + interceptors. (The GW2 **notes** interceptor is deliberately thin by contrast — no DynamoDB/STS, just caller-`sub` forwarding for row-level security; see `05b`.)

## Next Notebook

- **05b-deploy-notes-gateway.ipynb** (deploy GW2 + OpenSearch before the agent)

In [ ]:
# AWS Initialization - Load credentials and create session
from utils.notebook_init import init_aws

# This will:
# 1. Load credentials from .env file (if it exists)
# 2. Create and validate AWS session (env vars take precedence over SSO)
# 3. Return session, region, and account_id for use in this notebook
session, AWS_REGION, AWS_ACCOUNT_ID = init_aws()

# Initialize AWS clients
lambda_client = session.client("lambda", region_name=AWS_REGION)
ssm_client = session.client("ssm", region_name=AWS_REGION)

print("✅ Ready to proceed with AWS operations")
print(f"   Account ID: {AWS_ACCOUNT_ID}")
print(f"   Region: {AWS_REGION}")

## Step 1: Deploy Request Interceptor Lambda

The **REQUEST** interceptor runs *before* each tool call reaches the MCP server. It validates the JWT, authorizes the requested tool against the group's allow-list (DynamoDB role-map; disallowed → **403**), then maps the group → tenant IAM role, assumes it, and forwards the scoped credentials plus the caller's identity — so the downstream Athena query runs under that role's Lake Formation grants with the caller bound into its row predicate. This step also seeds the DynamoDB tenant role-map (group → allowed-tools and group → IAM role).

⏱️ Packages + deploys a Lambda (and seeds DynamoDB) — ~1–2 min.

In [ ]:
import subprocess
import sys

# Run deploy_interceptor.py to deploy Lambda function
# No capture_output: stdout/stderr stream live to the notebook so the packaging +
# deploy progress appears incrementally instead of all at once when the script ends.
result = subprocess.run(
    ["bash", "deploy.sh"],
    cwd="deployment/5a-gateway-setup/interceptor-request",
)

if result.returncode != 0:
    print("❌ Error: deploy.sh failed (see output above)")
else:
    print("\n✅ Interceptor Lambda deployed!")

## Step 2: Deploy Response Interceptor Lambda

The **RESPONSE** interceptor runs on the way back: it filters the `tools/list` result so each caller sees only the tools their group is allowed to use — the visible complement of the REQUEST interceptor's 403 enforcement.

⏱️ Packages + deploys a Lambda — ~1–2 min.

In [ ]:
# Deploy response interceptor Lambda
# No capture_output: stdout/stderr stream live to the notebook so the packaging +
# deploy progress appears incrementally instead of all at once when the script ends.
result = subprocess.run(
    ["bash", "deploy.sh"],
    cwd="deployment/5a-gateway-setup/interceptor-response",
)

if result.returncode != 0:
    print("❌ Error: deploy.sh failed (see output above)")
else:
    print("\n✅ Response Interceptor Lambda deployed!")

## Step 3: Get Required ARNs

Read the interceptor Lambda and MCP runtime ARNs from SSM — the gateway creation in Step 4 needs them.

In [ ]:
# IdP-agnostic prerequisites (both paths). These two ARNs are what GW1 wires up.

# REQUEST interceptor Lambda ARN from SSM (saved by interceptor-request/deploy.sh)
INTERCEPTOR_ARN = ssm_client.get_parameter(Name="/app/lakehouse-agent/interceptor-lambda-arn")["Parameter"]["Value"]
print(f"✅ Interceptor ARN: {INTERCEPTOR_ARN}")

# MCP Server Runtime ARN from SSM (the claims MCP server GW1 fronts)
MCP_SERVER_RUNTIME_ARN = ssm_client.get_parameter(Name="/app/lakehouse-agent/mcp-server-runtime-arn")["Parameter"][
    "Value"
]
print(f"✅ MCP Server ARN: {MCP_SERVER_RUNTIME_ARN}")

# Note: the authorizer config (Cognito user-pool / allowedClients vs Okta
# discovery / allowedAudience) is loaded and set inside create_gateway.py per
# IDP_PROVIDER — no IdP-specific inputs are needed here.

## Step 4: Create AgentCore Gateway

This creates GW1 and configures it with the MCP server plus the REQUEST + RESPONSE interceptors. `create_gateway.py` reads `IDP_PROVIDER` once and sets the authorizer accordingly:

- **## [COGNITO]** — `customJWTAuthorizer` with `allowedClients` (Cognito access tokens carry no `aud`, so validation is by client ID).
- **## [OKTA]** — `customJWTAuthorizer` with `allowedAudience` against the Okta custom-auth-server discovery URL.

The subprocess call below is the same on both paths.

⏱️ Gateway + target creation — ~1–2 min.

In [ ]:
# Create AgentCore Gateway
# No capture_output: stdout/stderr stream live to the notebook so the gateway +
# target creation progress appears incrementally instead of all at once at the end.
result = subprocess.run(
    [
        sys.executable,
        "create_gateway.py",
        "--yes",  # Auto-confirm for notebook execution
    ],
    cwd="deployment/5a-gateway-setup",
)

if result.returncode != 0:
    print("❌ Error: create_gateway.py failed (see output above)")
else:
    print("\n✅ Gateway created!")
    print("\n📋 Gateway ARN saved to SSM Parameter Store")

## Step 5: Verify Gateway Configuration

The create_gateway.py script automatically saves the Gateway ARN to SSM.
Run this cell to verify the deployment.

In [ ]:
# Verify Gateway configuration in SSM
print("Verifying Gateway configuration in SSM...\n")

parameters_to_check = [
    "/app/lakehouse-agent/gateway-arn",
    "/app/lakehouse-agent/gateway-id",
    "/app/lakehouse-agent/gateway-url",
]

all_found = True
for param_name in parameters_to_check:
    try:
        response = ssm_client.get_parameter(Name=param_name)
        value = response["Parameter"]["Value"]
        print(f"✅ {param_name}")
        print(f"   Value: {value}")
    except ssm_client.exceptions.ParameterNotFound:
        print(f"❌ {param_name} - NOT FOUND")
        all_found = False
    except Exception as e:
        print(f"⚠️  {param_name} - ERROR: {e}")
        all_found = False

if all_found:
    print("\n✅ Gateway configuration verified in SSM!")
else:
    print("\n⚠️  Gateway parameters missing.")
    print("    The create_gateway.py script should have saved these automatically.")
    print("    Check the deployment output for errors.")

## Summary

✅ **Gateway & Interceptor Deployment Complete!**

**What was created:**
- REQUEST Interceptor Lambda (JWT validation, caller-identity extraction)
- RESPONSE Interceptor Lambda (row/field filtering by user persona)
- AgentCore Claims Gateway (GW1, routing)

All configuration saved to SSM Parameter Store.

**Next Steps:**
Run **05b-deploy-notes-gateway.ipynb** (GW2 + OpenSearch), then **06-deploy-agent.ipynb**